In [1]:
import os
import sys

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("EcommerceDataPreparation")
    .master("local[2]")
    .config("spark.driver.memory", "12g")
    .config("spark.default.parallelism", "4")
    .config("spark.driver.maxResultSize", "1g")
    .config("spark.executor.memory", "4g")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.files.maxPartitionBytes", str(64 * 1024 * 1024))
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print("Spark version:", spark.version)
spark.range(5).show()

Spark version: 3.5.8
+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
+---+



In [7]:
# Process Sessions File In Batches
from pyspark.sql.functions import (
    col,
    size,
    to_timestamp,
    hour,
    dayofweek,
    when,
    lit
)

HDFS_SESSION_ROOT = "hdfs://localhost:9000/ecommerce/raw/sessions"
OUTPUT_PATH = "hdfs://localhost:9000/ecommerce/features/session_conversion"

TOTAL_FILES = 20
FILES_PER_BATCH = 4

In [8]:
raw_batch = (
    spark.read
    .schema(session_schema)
    .option("multiLine", "true")
    .json(batch_paths)
)

NameError: name 'session_schema' is not defined

In [ ]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
    DoubleType,
    ArrayType,
    MapType
)

geo_schema = StructType([
    StructField("city", StringType(), True),
    StructField("state", StringType(), True),
    StructField("country", StringType(), True),
    StructField("ip_address", StringType(), True)
])

device_schema = StructType([
    StructField("type", StringType(), True),
    StructField("os", StringType(), True),
    StructField("browser", StringType(), True)
])

page_view_schema = StructType([
    StructField("timestamp", StringType(), True),
    StructField("page_type", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("category_id", StringType(), True),
    StructField("view_duration", LongType(), True)
])

cart_item_schema = StructType([
    StructField("quantity", LongType(), True),
    StructField("price", DoubleType(), True)
])

session_schema = StructType([
    StructField("session_id", StringType(), True),
    StructField("user_id", StringType(), True),
    StructField("start_time", StringType(), True),
    StructField("end_time", StringType(), True),
    StructField("duration_seconds", LongType(), True),

    StructField("geo_data", geo_schema, True),

    StructField("device_profile", device_schema, True),

    StructField(
        "viewed_products",
        ArrayType(StringType()),
        True
    ),

    StructField(
        "page_views",
        ArrayType(page_view_schema),
        True
    ),

    # Product ID becomes the map key
    StructField(
        "cart_contents",
        MapType(
            StringType(),
            cart_item_schema
        ),
        True
    ),

    StructField("conversion_status", StringType(), True),
    StructField("referrer", StringType(), True)
])

In [6]:
# Process 4 session files at once

import gc
from pyspark import StorageLevel

first_batch = True

for start_index in range(0, TOTAL_FILES, FILES_PER_BATCH):

    end_index = min(
        start_index + FILES_PER_BATCH,
        TOTAL_FILES
    )

    batch_paths = [
        f"{HDFS_SESSION_ROOT}/sessions_{i}.json"
        for i in range(start_index, end_index)
    ]

    print(
        f"\nProcessing files "
        f"sessions_{start_index}.json "
        f"to sessions_{end_index - 1}.json"
    )

    # Read using the explicit schema
    raw_batch = (
        spark.read
        .schema(session_schema)
        .option("multiLine", "true")
        .json(batch_paths)
    )

    # Clean and engineer features
    prepared_batch = (
        prepare_session_batch(raw_batch)
        .repartition(8)
        .persist(StorageLevel.DISK_ONLY)
    )

    # Trigger processing once
    batch_count = prepared_batch.count()

    print(f"Prepared records: {batch_count:,}")

    write_mode = "overwrite" if first_batch else "append"

    (
        prepared_batch.write
        .mode(write_mode)
        .partitionBy("label")
        .parquet(OUTPUT_PATH)
    )

    print(
        f"Batch sessions_{start_index}.json "
        f"to sessions_{end_index - 1}.json saved successfully."
    )

    # Release current batch resources
    prepared_batch.unpersist()

    del raw_batch
    del prepared_batch

    spark.catalog.clearCache()
    gc.collect()

    first_batch = False

print("\nAll session batches were processed successfully.")


Processing files sessions_0.json to sessions_3.json


NameError: name 'session_schema' is not defined

In [4]:
# Cleaning And Feature-Engineering Function
from pyspark.sql.functions import (
    col,
    size,
    to_timestamp,
    hour,
    dayofweek,
    when,
    lit,
    coalesce
)

def prepare_session_batch(df):
    return (
        df
        .filter(col("session_id").isNotNull())
        .dropDuplicates(["session_id"])

        .withColumn(
            "session_start",
            to_timestamp("start_time")
        )
        .withColumn(
            "session_end",
            to_timestamp("end_time")
        )

        .filter(
            col("duration_seconds").isNotNull()
            & (col("duration_seconds") >= 0)
        )

        .withColumn(
            "viewed_product_count",
            coalesce(size("viewed_products"), lit(0))
        )
        .withColumn(
            "page_view_count",
            coalesce(size("page_views"), lit(0))
        )
        .withColumn(
            "cart_product_count",
            coalesce(size("cart_contents"), lit(0))
        )

        .withColumn(
            "session_hour",
            hour("session_start")
        )
        .withColumn(
            "session_day_of_week",
            dayofweek("session_start")
        )

        .withColumn(
            "device_type",
            col("device_profile.type")
        )
        .withColumn(
            "operating_system",
            col("device_profile.os")
        )
        .withColumn(
            "browser",
            col("device_profile.browser")
        )
        .withColumn(
            "country",
            col("geo_data.country")
        )

        .withColumn(
            "label",
            when(
                col("conversion_status") == "converted",
                lit(1.0)
            ).otherwise(lit(0.0))
        )

        .select(
            "session_id",
            "user_id",
            "session_start",
            "session_end",
            "duration_seconds",
            "viewed_product_count",
            "page_view_count",
            "cart_product_count",
            "session_hour",
            "session_day_of_week",
            "device_type",
            "operating_system",
            "browser",
            "country",
            "referrer",
            "conversion_status",
            "label"
        )
    )

In [ ]:
# Counting Before Writing Or Avoiding Duplicate
import gc
from pyspark import StorageLevel

first_batch = True

for start_index in range(0, TOTAL_FILES, FILES_PER_BATCH):
    end_index = min(
        start_index + FILES_PER_BATCH,
        TOTAL_FILES
    )

    batch_paths = [
        f"{HDFS_SESSION_ROOT}/sessions_{i}.json"
        for i in range(start_index, end_index)
    ]

    print(
        f"\nBatch: {start_index} to {end_index - 1}"
    )

    raw_batch = spark.read.json(batch_paths)

    prepared_batch = (
        prepare_session_batch(raw_batch)
        .repartition(8)
        .persist(StorageLevel.DISK_ONLY)
    )

    batch_count = prepared_batch.count()

    print(f"Records prepared: {batch_count:,}")

    write_mode = "overwrite" if first_batch else "append"

    (
        prepared_batch.write
        .mode(write_mode)
        .partitionBy("label")
        .parquet(OUTPUT_PATH)
    )

    prepared_batch.unpersist()

    del raw_batch
    del prepared_batch

    spark.catalog.clearCache()
    gc.collect()

    first_batch = False

In [ ]:
# Read Prepared Result
session_features_df = spark.read.parquet(
    "hdfs://localhost:9000/ecommerce/features/session_conversion"
)
print(
    "Prepared sessions:",
    session_features_df.count()
)

session_features_df.printSchema()